# MicroFinML - Model Training
## Phase 3: Train XGBoost, Random Forest, and Logistic Regression Models

**Models:**
1. **Logistic Regression** - Baseline model
2. **Random Forest** - Ensemble method
3. **XGBoost** - Primary model (state-of-the-art for tabular data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

from src.model_training import get_models, train_model, cross_validate_model, train_all_models

print('Modules loaded successfully!')

## 1. Load Processed Data

In [ ]:
processed_dir = '../data/processed'

X_train = pd.read_csv(os.path.join(processed_dir, 'X_train.csv'))
X_val = pd.read_csv(os.path.join(processed_dir, 'X_val.csv'))
X_test = pd.read_csv(os.path.join(processed_dir, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(processed_dir, 'y_train.csv')).squeeze()
y_val = pd.read_csv(os.path.join(processed_dir, 'y_val.csv')).squeeze()
y_test = pd.read_csv(os.path.join(processed_dir, 'y_test.csv')).squeeze()

print(f'Training set:   {X_train.shape[0]:,} samples, {X_train.shape[1]} features')
print(f'Validation set: {X_val.shape[0]:,} samples')
print(f'Test set:       {X_test.shape[0]:,} samples')
print(f'\nTraining default rate: {y_train.mean():.4f}')
print(f'Validation default rate: {y_val.mean():.4f}')
print(f'Test default rate: {y_test.mean():.4f}')

## 2. Train All Models

In [ ]:
results = train_all_models(
    X_train, y_train,
    tune=False,
    save_dir='../models'
)

## 3. Cross-Validation Results Summary

In [ ]:
print('=' * 60)
print('CROSS-VALIDATION RESULTS SUMMARY')
print('=' * 60)

cv_data = []
for name, res in results.items():
    cv_data.append({
        'Model': name,
        'CV ROC-AUC (Mean)': round(res['cv_roc_auc_mean'], 4),
        'CV ROC-AUC (Std)': round(res['cv_roc_auc_std'], 4),
        'Training Time (s)': round(res['train_time'], 2)
    })

cv_df = pd.DataFrame(cv_data).set_index('Model')
cv_df

In [ ]:
# Visualize CV scores
fig, ax = plt.subplots(figsize=(10, 5))

model_names = list(results.keys())
cv_means = [results[n]['cv_roc_auc_mean'] for n in model_names]
cv_stds = [results[n]['cv_roc_auc_std'] for n in model_names]
colors = ['#2196F3', '#4CAF50', '#FF5722']

bars = ax.bar(model_names, cv_means, yerr=cv_stds, capsize=8,
              color=colors, edgecolor='black', alpha=0.85)
ax.set_title('Cross-Validation ROC-AUC Scores', fontsize=14, fontweight='bold')
ax.set_ylabel('ROC-AUC Score')
ax.set_ylim(0, 1.05)

for bar, mean, std in zip(bars, cv_means, cv_stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
            f'{mean:.4f}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../results/figures/model_comparison/cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nModels trained and saved to ../models/')
print('Proceed to 04_model_evaluation.ipynb for detailed evaluation.')